In [21]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 0 : CONFIGURATION                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

from pyspark.sql import functions as F
from pyspark.sql.types import *

storage_account = "energybigdatastorage"
container_raw = "raw"
container_processed = "processed"

path_raw = f"abfss://{container_raw}@{storage_account}.dfs.core.windows.net/energy_data_extracted/archive (3).zip/hhblock_dataset/hhblock_dataset"
path_processed = f"abfss://{container_processed}@{storage_account}.dfs.core.windows.net/hhblock_dataset/"

print(f"Source: {path_raw}")
print(f"Destination: {path_processed}")

In [22]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 1 : INGESTION                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("dateFormat", "yyyy-MM-dd") \
    .load(path_raw)

print(f"Nombre de lignes: {df.count()}")
print(f"Nombre de colonnes: {len(df.columns)}")
print(f"Premieres colonnes: {df.columns[:10]}")
print(f"\n=== SCHEMA ===")
df.printSchema()
print(f"\n=== APERCU ===")
df.show(3)

In [23]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 2 : PROFILING AVANT NETTOYAGE                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print("=== VALEURS NULLES (premieres colonnes) ===")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns[:12]]).show()

print(f"\n=== DOUBLONS ===")
print(f"Nombre de doublons: {df.count() - df.dropDuplicates().count()}")

# Identifier les colonnes hh_
hh_cols = [c for c in df.columns if c.startswith("hh_")]
print(f"\n=== COLONNES DEMI-HEURE ===")
print(f"Nombre de colonnes hh_: {len(hh_cols)}")
print(f"Exemples: {hh_cols[:5]}")

# Verifier quelques valeurs
print(f"\n=== APERCU COLONNES hh_ ===")
df.select("LCLid", "day", *hh_cols[:5]).show(5)

In [24]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 3 : NETTOYAGE                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# 1. Nettoyer LCLid et day
df_clean = df \
    .withColumn("LCLid", F.upper(F.trim(F.col("LCLid")))) \
    .withColumn("day", F.to_date(F.col("day"), "yyyy-MM-dd")) \
    .filter(F.col("LCLid").isNotNull() & F.col("day").isNotNull())

# 2. Supprimer doublons
df_clean = df_clean.dropDuplicates(["LCLid", "day"])

# 3. Convertir hh_ en double, negatifs -> NULL
for c in hh_cols:
    df_clean = df_clean.withColumn(c, 
        F.when(F.col(c).cast("double") < 0, F.lit(None))
        .otherwise(F.col(c).cast("double"))
    )

# 4. Chaines vides / "null" / "nan" -> NULL
for c in hh_cols:
    df_clean = df_clean.withColumn(c,
        F.when(
            (F.col(c).isNull()) | 
            (F.trim(F.col(c).cast("string")) == "") |
            (F.trim(F.lower(F.col(c).cast("string"))).isin("null", "nan")),
            F.lit(None)
        ).otherwise(F.col(c))
    )

# 5. Compteur de valeurs manquantes par ligne
df_clean = df_clean.withColumn("missing_hh_count", 
    sum(F.when(F.col(c).isNull(), F.lit(1)).otherwise(F.lit(0)) for c in hh_cols)
)

# 6. Supprimer lignes avec >75% de valeurs manquantes
threshold = int(len(hh_cols) * 0.75)
removed_missing = df_clean.filter(F.col("missing_hh_count") > threshold).count()
df_clean = df_clean.filter(F.col("missing_hh_count") <= threshold)
print(f"Lignes supprimees (>75% manquantes): {removed_missing}")

# 7. Consommation journaliere totale
df_clean = df_clean.withColumn("daily_total", 
    sum(F.coalesce(F.col(c), F.lit(0.0)) for c in hh_cols)
)

# 8. Features temporelles
df_clean = df_clean \
    .withColumn("year", F.year(F.col("day"))) \
    .withColumn("month", F.month(F.col("day"))) \
    .withColumn("dayofweek", F.dayofweek(F.col("day"))) \
    .withColumn("is_weekend", F.when(F.col("dayofweek").isin([1, 7]), 1).otherwise(0)) \
    .withColumn("quarter", F.quarter(F.col("day"))) \
    .withColumn("processed_date", F.current_date())

print(f"\nNombre de lignes nettoyees: {df_clean.count()}")
df_clean.select("LCLid", "day", "missing_hh_count", "daily_total", "year", "month").show(5)

In [25]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 4 : STATISTIQUES APRES NETTOYAGE                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print("=== DISTRIBUTION VALEURS MANQUANTES ===")
df_clean.groupBy("missing_hh_count").count().orderBy("missing_hh_count").show(20)

print("\n=== STATISTIQUES DAILY_TOTAL ===")
df_clean.select(
    F.min("daily_total").alias("min"),
    F.max("daily_total").alias("max"),
    F.avg("daily_total").alias("moyenne"),
    F.stddev("daily_total").alias("ecart_type")
).show()

print("\n=== DISTRIBUTION TEMPORRELLE ===")
df_clean.select(
    F.min("day").alias("date_min"),
    F.max("day").alias("date_max"),
    F.countDistinct("day").alias("nb_jours"),
    F.countDistinct("LCLid").alias("nb_compteurs")
).show()

In [26]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 5 : SAUVEGARDE DELTA                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("year", "month") \
    .save(path_processed)

print("Sauvegarde terminee dans processed/hhblock_dataset/")

In [27]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 6 : VERIFICATION                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

df_verify = spark.read.format("delta").load(path_processed)
print(f"Verification: {df_verify.count()} lignes")
print(f"\n=== SCHEMA ===")
df_verify.printSchema()
print(f"\n=== APERCU ===")
df_verify.select("LCLid", "day", "missing_hh_count", "daily_total", "year", "month").show(5)

print(f"\n=== TESTS RAPIDES ===")
print(f"LCLid NULL: {df_verify.filter(F.col('LCLid').isNull()).count()}")
print(f"day NULL: {df_verify.filter(F.col('day').isNull()).count()}")
print(f"Negatifs (hh_0): {df_verify.filter(F.col('hh_0') < 0).count()}")
print(f"Daily_total NULL: {df_verify.filter(F.col('daily_total').isNull()).count()}")